In [1]:
from bson import ObjectId
from flatten_json import flatten
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import datetime as dt
import re
import pymongo
import os
import sqlalchemy
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from google.cloud import bigquery

In [2]:
env_path = Path('/home/aurora') / '.env'
load_dotenv(dotenv_path=env_path)

True

In [149]:
date = str(dt.datetime.now().date() - dt.timedelta(1))
date_range_list = [str(dt.datetime.now().date() - dt.timedelta(i)) for i in range(1,5)]

In [4]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (f"""SELECT count(distinct(user_id)) FROM `hitwicketsuperstars.analytics_190927423.events_intraday_{date.replace('-','')}` where event_name='screen_view'""")
dau = client.query(query).to_dataframe().iloc[0,0]

In [7]:
bq_table_str = (f"hitwicketsuperstars.analytics_190927423.events_intraday_{date.replace('-','')}")
query = (
    f"""SELECT
  user_id,
  MIN(event_timestamp),
  platform
FROM
  `{bq_table_str}`
WHERE app_info.id = 'cricketgames.hitwicket.strategy'
GROUP BY
  user_id,
  platform"""
)
new_devices_today = client.query(query).to_dataframe()
query = (
    f"""SELECT
    * FROM
    `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
)
new_user_reference = client.query(query).to_dataframe()

new_devices_today = new_devices_today[~new_devices_today["user_id"].isin(new_user_reference["device_id"])]
new_devices_today.columns = ['device_id', 'user_first_touch_timestamp','platform']
new_devices_today['user_first_touch_timestamp'] = pd.to_datetime(new_devices_today['user_first_touch_timestamp'], unit = 'us')
new_devices_today['user_first_touch_timestamp'] = new_devices_today['user_first_touch_timestamp'].astype('datetime64[s]')

In [11]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
db = cursor.superstars
aw = list(db.users.find({}, {'login_details.last_request_at', 'sign_up_details.device.id', 'created_at'}))
all_users = pd.DataFrame([flatten(d) for d in aw])
print(len(all_users))
all_users = all_users.sort_values(['sign_up_details_device_id','created_at'])
all_users = all_users.drop_duplicates('sign_up_details_device_id')
print(len(all_users))
the_merge = pd.merge(new_devices_today, all_users, left_on='device_id', right_on='sign_up_details_device_id', how = 'left')
the_merge['time_diff'] = the_merge['login_details_last_request_at'] - the_merge['user_first_touch_timestamp']
the_merge['d1'] = (the_merge['time_diff'] >= pd.Timedelta('1 days 00:00:00')).astype(int)
the_merge['date'] = the_merge['user_first_touch_timestamp'].dt.date

135430
131027


In [62]:
android = the_merge[the_merge.platform == 'ANDROID']
ios = the_merge[the_merge.platform == 'IOS']

In [63]:
and_grouped = android.groupby('date').agg({'device_id' : 'count', '_id' : 'count'})
ios_grouped = ios.groupby('date').agg({'device_id' : 'count', '_id' : 'count'})

In [12]:
end = dt.datetime.today()
end = end.replace(hour=18, minute=30, second=0, microsecond=0)
start = end - dt.timedelta(5)

In [13]:
database_username = os.environ['localuser']
database_password = os.environ['pass']
database_ip       = 'localhost'
database_name     = os.environ['dbname']
database_connection = sqlalchemy.create_engine(('mysql+pymysql://{0}:{1}@{2}/{3}'.format(database_username, database_password,database_ip, database_name)))
local_db = database_connection.connect()

In [34]:
android_retention = pd.read_sql('select * from android_retention_lr',local_db)
android_retention = android_retention.sort_values('date', ascending = False)
android_retention['create_team%'] = (100*android_retention['users']/android_retention['opened_app']).round(1)
android_retention['d1%'] = (100*android_retention['d1']/android_retention['opened_app']).round(1)
android_retention5 = android_retention[["date","opened_app","users","create_team%","d1","d1%"]]
android_retention5['date'] = android_retention5['date'].astype(str)
# print(android_retention5)
opened_app = android_retention5[android_retention5['date'] == str(date)]
# print(opened_app)
opened_app = opened_app.iloc[0]['opened_app']
android_retention5 = android_retention5[android_retention5['date'].isin(date_range_list)]
android_retention5.columns = ["date","New Installs","Created Team","Created Team %","D1 Retained","D1%"]
android_retention5.set_index('date',inplace=True)

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


In [91]:
latest = and_grouped.iloc[-1]
latest['team_create_perc'] = ((latest['_id']/latest['device_id'])*100).round(1)
latest = latest.tolist()
latest+=[0,0]
android_retention5.loc[str(dt.date.today())] = latest
android_retention5 = android_retention5.sort_index(ascending=False)

In [97]:
android_retention5

,New Installs,Created Team,Created Team %,D1 Retained,D1%
date,,,,,
2019-06-20,264.0,102.0,38.6,0.0,0.0
2019-06-19,471.0,196.0,41.6,1.0,0.2
2019-06-18,508.0,223.0,43.9,21.0,4.1
2019-06-17,356.0,142.0,39.9,12.0,3.4
2019-06-16,346.0,107.0,30.9,24.0,6.9


In [72]:
ios_retention = pd.read_sql('select * from ios_retention_lr',local_db)
ios_retention = ios_retention.sort_values('date', ascending = False)
ios_retention['create_team%'] = (100*ios_retention['users']/ios_retention['opened_app']).round(1)
ios_retention['d1%'] = (100*ios_retention['d1']/ios_retention['opened_app']).round(1)
ios_retention5 = ios_retention[["date","opened_app","users","create_team%","d1","d1%"]]
ios_retention5['date'] = ios_retention5['date'].astype(str)
opened_app = ios_retention5[ios_retention5['date'] == str(date)]
# print(opened_app)
opened_app = opened_app.iloc[0]['opened_app']
ios_retention5 = ios_retention5[ios_retention5['date'].isin(date_range_list)]
ios_retention5.columns = ["date","New Installs","Created Team","Created Team %","D1 Retained","D1%"]
ios_retention5.set_index('date',inplace=True)

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


In [98]:
latest = ios_grouped.iloc[-1]
latest['team_create_perc'] = ((latest['_id']/latest['device_id'])*100).round(1)
latest = latest.tolist()
latest+=[0,0]
ios_retention5.loc[str(dt.date.today())] = latest
ios_retention5 = ios_retention5.sort_index(ascending=False)

In [99]:
ios_retention5

,New Installs,Created Team,Created Team %,D1 Retained,D1%
date,,,,,
2019-06-20,8.0,2.0,25.0,0.0,0.0
2019-06-19,17.0,9.0,52.9,0.0,0.0
2019-06-18,20.0,6.0,30.0,2.0,10.0
2019-06-17,18.0,2.0,11.1,0.0,0.0
2019-06-16,15.0,5.0,33.3,1.0,6.7


In [100]:
def d1_color_setter(value):
    if value <= 4:
        color = 'red'
    elif value >= 8:
        color = 'green'
    else:
        color = 'none'
    return 'color: %s' % color

In [101]:
def ct_color_setter(value):
    if value <= 20:
        color = 'red'
    elif value >= 30:
        color = 'green'
    else:
        color = 'none'
    return 'color: %s' % color

In [102]:
def bold(value):
    return 'font-weight: bold;'

In [3]:
html_str = """<html>
<head>
<style>

    h2 {
        font-family: Helvetica, Arial, sans-serif;
    }
    table, th, td {
        border: 1px solid black;
        border-collapse: collapse;
    }
    th, td {
        padding: 5px;
        font-family: Helvetica, Arial, sans-serif;
        font-size: 100%;
    }
    tbody tr:nth-child(odd) {background: #eee}
    tbody tr:nth-child(even) {background: #fff}
    table tbody tr td:hover {
        background-color: #fcffb2;
    }
    .col_heading{
        font-weight: normal;
    }
    .row0, .row1{
        font-weight: normal;
    }
    .row_heading, .blank{
        display: none;
    }
    .row0, .row2{
        background-color: #dddddd;
    }
    .row0:hover, .row2:hover{
        background-color: #fcffb2;
    }
    .row2{
        font-weight: bold;
    }
    .col1, .col2, .col3, .col4, .col5{
        text-align: right;
    }
</style>
</head>
<body>
"""

In [151]:
html_str+=f""" 
<img src="https://d8tuj5f40nouo.cloudfront.net/images/web/landing/logo.png" width="200" height="83">
<h2>Superstars: {str((dt.datetime.now().date()).strftime('%A, %B %d'))}</h2>
<h3>Daily Active Users: {dau}</h3>
"""

In [152]:
date_range_list+=[str(dt.date.today())]
date_range_list=sorted(date_range_list)[::-1]
for i in range(len(date_range_list)):
    curr_date = date_range_list[i]
    l=[]
    l.append(['Android'] + list(android_retention5.loc[curr_date]))
    l.append(['IOS'] + list(ios_retention5.loc[curr_date]))
    df = pd.DataFrame(l, columns = [dt.datetime.strptime(curr_date,'%Y-%m-%d').strftime('%A, %d %b')]+list(android_retention5.columns))
    total_opened = sum(df['New Installs'].tolist())
    total_created = sum(df['Created Team'].tolist())
    total_created_perc = round((100*total_created/total_opened),1)
    total_d1 = sum(df['D1 Retained'].tolist())
    total_d1_perc = round((100*total_d1/total_opened),1)
    df.loc[len(df)] = (['Total'] + [total_opened,total_created,total_created_perc,total_d1,total_d1_perc])
    df['New Installs'] = df['New Installs'].apply(lambda x: int(x))
    df['Created Team'] = df['Created Team'].apply(lambda x: int(x))
    df['D1 Retained'] = df['D1 Retained'].apply(lambda x: int(x))
    if(i==0):
        df = df[[df.columns[0],"New Installs", "Created Team", "Created Team %"]]
        signups = str(int(df.iloc[len(df)-1]['New Installs'])) + '(' + str(int(df.iloc[len(df)-1]['Created Team'])) + ')'
        df = df.style.applymap(ct_color_setter,subset=['Created Team %'])
    else:
        df = df.style.applymap(d1_color_setter,subset=['D1%']).applymap(ct_color_setter,subset=['Created Team %'])
    df_str = df.render(index = False).replace('\n','')
    html_str+= f"""
        { df_str }
        <br><br>
    """
html_str+="<a href=\"https://console.firebase.google.com/u/0/project/hitwicketsuperstars/overview\">Firebase Console</a></body></html>"

In [153]:
html_str = html_str.replace('\n','')

In [5]:
subject = f"""Superstars Users: {signups} - {str((dt.datetime.now().date()).strftime('%B %d'))}"""
gmail_user = os.environ['mail']
gmail_password = os.environ['mail_token']
# to = ['analytics@hitwicket.com','product@hitwicket.com','growth@hitwicket.com']
to = ['mihir@hitwicket.com']
# cc = ['anshaj@hitwicket.com','yash@hitwicket.com','aruwin@hitwicket.com']
sent_from = gmail_user
text = "Please use an html reader"
message = MIMEMultipart("alternative", None, [MIMEText(text), MIMEText(html_str,'html')])
message['To'] = ','.join(to)
message['From'] = "Analytics <" + os.environ['mail'] + ">"
# message['Bcc'] = 'mihir@hitwicket.com'
message['Subject'] = subject

In [6]:
s = smtplib.SMTP('smtp.gmail.com', 587)
s.starttls()
s.login(gmail_user, gmail_password)
s.sendmail(sent_from, to, message.as_string())
s.quit()

(221, b'2.0.0 closing connection o5sm14554883iob.7 - gsmtp')